## Phase 3: RNNs & Sequential Memory (Air Quality)

In [10]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, Subset
from torch import nn
import os
import pandas as pd
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

Using device: mps


### Dataset and Utility Classes

In [4]:
def is_it_nan(val):
    """Utility to check if a value is NaN, since NaN != NaN."""
    return val != val


class TimeSeriesDataset(Dataset):
    """
    The Dataset definition for time series data specifically delhi_aqi.csv
    """

    def __init__(self, file_path, seq_len, hop, col_name='pm2_5', classify=False, all_features=False, train_frac=0.7):
        df = pd.read_csv(file_path)

        if all_features:
            feature_cols = [c for c in df.columns if c != 'date']
            data = torch.tensor(df[feature_cols].values, dtype=torch.float32)  # (T, F)
            pm2_5_idx = feature_cols.index(col_name)
            self.input_dim = seq_len * len(feature_cols)
            self.num_features = len(feature_cols)
        else:
            data = torch.tensor(df[col_name].values, dtype=torch.float32)      # (T,)
            pm2_5_idx = None
            self.input_dim = seq_len
            self.num_features = 1

        # Normalise using training portion stats only — prevents leakage into val/test
        n_train = int(train_frac * len(data))
        self.X_mean = data[:n_train].mean(dim=0)
        self.X_std  = data[:n_train].std(dim=0).clamp(min=1e-8)
        data = (data - self.X_mean) / self.X_std

        # PM2.5 stats for denormalising predictions
        self.y_mean = self.X_mean[pm2_5_idx] if all_features else self.X_mean
        self.y_std  = self.X_std[pm2_5_idx]  if all_features else self.X_std

        self.X = []
        self.y = []

        for i in range(len(data) - seq_len - hop):
            x_seq = data[i:i+seq_len]
            y_target = data[i+seq_len+hop, pm2_5_idx] if all_features else data[i+seq_len+hop]

            self.X.append(x_seq)  # shape: (seq_len, F) or (seq_len,)
            if classify:
                self.y.append(y_target > (200 - self.y_mean) / self.y_std)  # 200 threshold in normalised space
            else:
                self.y.append(y_target)

        self.X = torch.stack(self.X)
        self.y = torch.stack(self.y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [5]:
def train_loop(train_dataloader, val_dataloader, model, loss_fn, optimizer, max_iter=100, patience=10):
    """
    Basic training Loop.
    Early stopping: halts if validation loss does not improve for `patience` consecutive epochs.
    """
    best_val_loss = float('inf')
    epochs_no_improve = 0
    best_model_state = None

    for iter in range(max_iter):
        train_loss = 0
        model.train()
        for batch, (X, y) in enumerate(train_dataloader):
            X, y = X.to(device), y.float().to(device)
            pred = model(X)
            loss = loss_fn(pred, y)

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            train_loss += loss

        train_loss /= len(train_dataloader)

        test_loss = 0
        model.eval()
        with torch.no_grad():
            for X, y in val_dataloader:
                X, y = X.to(device), y.float().to(device)
                pred = model(X)
                test_loss += loss_fn(pred, y).item()

        test_loss /= len(val_dataloader)

        if iter % 10 == 0:
            print(f"Epoch {iter}: Training Loss: {train_loss:.4f}, Validation Loss: {test_loss:.4f}")

        if test_loss < best_val_loss:
            best_val_loss = test_loss
            epochs_no_improve = 0
            best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping at epoch {iter}. Best Validation Loss: {best_val_loss:.4f}")
                model.load_state_dict(best_model_state)
                return None

    return None


def test_loop(dataloader, model, loss_fn):
    """
    Basic Test Loop.
    """
    model.eval()
    test_loss = 0
    pred_list = []
    y_list = []

    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.float().to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            pred_list.extend(pred.tolist())
            y_list.extend(y.tolist())

    test_loss /= len(dataloader)
    print(f"Test Loss (MSE): {test_loss:.4f}")
    return pred_list, y_list

In [6]:
def train_loop_rnn(train_dataloader, model, loss_fn, optimizer):
    """
    Basic training Loop for exploding gradients demo (Problem 3.9).
    Trains until NaN loss is detected, then restores the last valid state.
    Returns the largest singular value of the recurrent weight matrix.
    """
    loss = torch.tensor(0.0)
    prev_state = {k: v.clone() for k, v in model.state_dict().items()}
    i = 0

    while not is_it_nan(loss.item()):
        prev_state = {k: v.clone() for k, v in model.state_dict().items()}
        train_loss = 0
        model.train()
        for batch, (X, y) in enumerate(train_dataloader):
            X, y = X.to(device), y.float().to(device)
            pred = model(X)
            loss = loss_fn(pred, y)

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            train_loss += loss

        train_loss /= len(train_dataloader)
        i += 1
        print(f"Epoch Number:{i}")

    model.load_state_dict(prev_state)
    svd = torch.linalg.svd(model.linearh.weight.detach().cpu())
    return svd.S[0].item()

In [7]:
class OwnVanillaRNN(nn.Module):

    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()

        self.hidden_dim = hidden_dim

        self.linearx = nn.Linear(input_dim, hidden_dim)
        self.linearh = nn.Linear(hidden_dim, hidden_dim)
        self.lineary = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(-1)
        if x.dim() == 1:
            x = x.unsqueeze(0)
            x = x.unsqueeze(-1)
        batch_size, seq_length, _ = x.shape

        h_t = torch.zeros(batch_size, self.hidden_dim, device=x.device)

        for t in range(seq_length):
            x_t = x[:, t, :]
            h_t = torch.tanh(self.linearx(x_t) + self.linearh(h_t))

        y = self.lineary(h_t)
        return y.squeeze(1)

    def bptt_decay(self, seq_len=100, input_dim=1):
        # MPS does not support retain_grad() on intermediate tensors — run on CPU
        cpu = torch.device('cpu')
        self.to(cpu)

        hidden_dim = self.hidden_dim
        x = torch.randn(1, seq_len, input_dim, device=cpu)

        h_t = torch.zeros(1, hidden_dim, device=cpu)
        hidden_states = {}

        for t in range(seq_len):
            x_t = x[:, t, :]
            h_t = torch.tanh(self.linearx(x_t) + self.linearh(h_t))
            h_t.retain_grad()
            hidden_states[t] = h_t

        y = self.lineary(h_t)
        target = torch.zeros(1, device=cpu)
        loss = nn.MSELoss()(y.squeeze(1), target)

        loss.backward()
        grad_t100 = hidden_states[99].grad.norm().item()
        grad_t50  = hidden_states[49].grad.norm().item()
        grad_t0   = hidden_states[0].grad.norm().item()

        print(f"||dL/dh|| at t=100: {grad_t100:.2e}")
        print(f"||dL/dh|| at t=50:  {grad_t50:.2e}")
        print(f"||dL/dh|| at t=0:   {grad_t0:.2e}")

        self.to(device)
        return hidden_states

### Dataset Setup

In [8]:
time_series_file_path = os.path.join('..', 'Datasets', 'delhi_aqi_dataset', 'delhi_aqi.csv')

data_regression = TimeSeriesDataset(time_series_file_path, 72, 24)

#### **Problem 3.7**: Vanilla RNN from scratch

In [11]:
torch.manual_seed(42)
torch.mps.manual_seed(42)

N = len(data_regression)

train_indices = list(range(int(0.7*N)))
val_indices   = list(range(int(0.7*N), int(0.85*N)))
test_indices  = list(range(int(0.85*N), N))

train_dataset = Subset(data_regression, train_indices)
val_dataset   = Subset(data_regression, val_indices)
test_dataset  = Subset(data_regression, test_indices)

train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=False)
val_dataloader   = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_dataloader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

rnn_model = OwnVanillaRNN(1, 64, 1).to(device)
learning_rate = 1e-3
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(rnn_model.parameters(), lr=learning_rate)
epochs = 100
print("Entering training loop")
train_loop(train_dataloader, val_dataloader, rnn_model, loss_fn, optimizer, epochs, patience=10)
print("Entering test loop")
_,_ = test_loop(test_dataloader, rnn_model, loss_fn)

Entering training loop
Epoch 0: Training Loss: 0.6870, Validation Loss: 0.2424
Epoch 10: Training Loss: 0.7057, Validation Loss: 0.2641
Early stopping at epoch 11. Best Validation Loss: 0.2412
Entering test loop
Test Loss (MSE): 0.9237


#### **Problem 3.8**: BPTT Decay

In [12]:
rnn_model = OwnVanillaRNN(1, 64, 1).to(device)
hidden_states = rnn_model.bptt_decay()

||dL/dh|| at t=100: 4.71e-01
||dL/dh|| at t=50:  6.76e-21
||dL/dh|| at t=0:   0.00e+00


#### **Problem 3.9**: Exploding Gradients

In [13]:
N = len(data_regression)

train_indices = list(range(int(0.7*N)))
val_indices   = list(range(int(0.7*N), int(0.85*N)))
test_indices  = list(range(int(0.85*N), N))

train_dataset = Subset(data_regression, train_indices)
val_dataset   = Subset(data_regression, val_indices)
test_dataset  = Subset(data_regression, test_indices)

train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=False)
val_dataloader   = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_dataloader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

rnn_model = OwnVanillaRNN(1, 64, 1).to(device)
learning_rate = 10
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(rnn_model.parameters(), lr=learning_rate)
epochs = 100
largest_singular_value = train_loop_rnn(train_dataloader, rnn_model, loss_fn, optimizer)
print(f"Largest Singular Value of Recurrent Weight Matrix during Training: {largest_singular_value:.4f}")

Epoch Number:1
Largest Singular Value of Recurrent Weight Matrix during Training: 1.1760
